# 2. YAMNet Top-5 Classification

This notebook is built specifically to run **after the current**
`1. Audio_download_and_classification.ipynb`.

Notebook 1 already:

- downloads each recording into `../data/audio/`;
- keeps the audio in its original cached format (`.mp3`, `.m4a`, `.wav`, `.ogg`, `.aac`, or `.flac`);
- creates `../data/audio_speech_labels.csv`; and
- adds the existing `is_speech` speech/non-speech label.

This notebook **does not redo speech/non-speech classification**.

It runs Google's YAMNet model on every row available in
`audio_speech_labels.csv`, computes the **top 5 YAMNet classes**, and writes
the results to a **new CSV**:

`../data/yamnet_top5_classifications.csv`

The original `audio_speech_labels.csv` is never overwritten.

### New YAMNet columns

For each audio clip the CSV contains:

- `yamnet_rank_1_class` ... `yamnet_rank_5_class`
- `yamnet_rank_1_mean_score` ... `yamnet_rank_5_mean_score`
- `yamnet_rank_1_max_score` ... `yamnet_rank_5_max_score`
- `yamnet_status`
- `yamnet_error`
- `yamnet_frame_count`
- `yamnet_audio_seconds_used`

All original columns, including `is_speech`, are retained in the new CSV.

## 1. Install YAMNet dependencies

Run this cell once in the same `.venv` you use for Notebook 1.

If VS Code asks you to restart the kernel after installation, restart it and
continue from the next cell.

In [1]:
%pip install -U tensorflow tensorflow-hub


[notice] A new release of pip is available: 25.3 -> 26.2
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Imports

In [2]:
from __future__ import annotations

import csv
import json
import os
import re
import subprocess
import time
from pathlib import Path
from typing import Any
from urllib.parse import urlparse

import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow_hub as hub
from tqdm.auto import tqdm


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tensorflow_hub/__init__.py:61: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import parse_version


## 3. Configuration

The defaults below match Notebook 1.

The path resolver works whether VS Code starts the notebook from the project
root or from the `jupyter_notebooks/` directory.

In [3]:
def find_data_dir() -> Path:
    """
    Locate the project's data directory robustly.

    Notebook 1 normally uses ../data when run from jupyter_notebooks/.
    VS Code can sometimes start a notebook with the project root as cwd,
    so both layouts are supported.
    """
    cwd = Path.cwd().resolve()

    candidates = [
        cwd / "data",
        cwd.parent / "data",
        Path("../data").resolve(),
    ]

    for candidate in candidates:
        if (candidate / "audio_speech_labels.csv").exists():
            return candidate

    # Return Notebook 1's normal location so the eventual error is explicit.
    return Path("../data").resolve()


DATA_DIR = find_data_dir()
AUDIO_DIR = DATA_DIR / "audio"

INPUT_CSV = DATA_DIR / "audio_speech_labels.csv"

# IMPORTANT: this is a NEW CSV. The input CSV is never overwritten.
OUTPUT_CSV = DATA_DIR / "yamnet_top5_classifications.csv"

# Match Notebook 1's configurable column names where possible.
ID_COLUMN = os.getenv("ID_COLUMN", "id")
AUDIO_URL_COLUMN = os.getenv("AUDIO_URL_COLUMN", "streamableUrl")
SPEECH_COLUMN = "is_speech"

TARGET_SAMPLE_RATE = 16_000
TOP_K = 5

# Save progress every N newly processed clips.
SAVE_EVERY = 20

# None = run the entire pending dataset.
# Set to e.g. 10 for a quick test.
LIMIT = None

# If False, rows already saved as failed are not retried.
# Change to True if you fix an audio issue and want to retry failures.
RETRY_FAILED = False

# Maximum time allowed for ffmpeg to decode one local file.
FFMPEG_TIMEOUT_SECONDS = 600

YAMNET_HANDLE = "https://tfhub.dev/google/yamnet/1"

print(f"Data directory: {DATA_DIR}")
print(f"Audio directory: {AUDIO_DIR}")
print(f"Input CSV:       {INPUT_CSV}")
print(f"NEW output CSV:  {OUTPUT_CSV}")
print(f"Top-K:           {TOP_K}")


Data directory: /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data
Audio directory: /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/audio
Input CSV:       /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/audio_speech_labels.csv
NEW output CSV:  /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/yamnet_top5_classifications.csv
Top-K:           5


## 4. Validate the output from Notebook 1

This checks that the speech/non-speech CSV exists and that the required
columns are present.

In [4]:
if not INPUT_CSV.exists():
    raise FileNotFoundError(
        f"Could not find: {INPUT_CSV}\n"
        "Run Notebook 1 through Stage 3 first so that "
        "audio_speech_labels.csv exists."
    )

if not AUDIO_DIR.exists():
    raise FileNotFoundError(
        f"Could not find the audio cache directory: {AUDIO_DIR}\n"
        "Run Notebook 1 Stage 2 first."
    )

source_df = pd.read_csv(INPUT_CSV)

if ID_COLUMN not in source_df.columns:
    raise KeyError(
        f"ID column '{ID_COLUMN}' was not found.\n"
        f"Available columns: {list(source_df.columns)}"
    )

if AUDIO_URL_COLUMN not in source_df.columns:
    # Helpful fallback in case .env is not loaded in this kernel.
    if "streamableUrl" in source_df.columns:
        AUDIO_URL_COLUMN = "streamableUrl"
        print(
            "AUDIO_URL_COLUMN was not found from the environment; "
            "using 'streamableUrl'."
        )
    else:
        raise KeyError(
            f"Audio URL column '{AUDIO_URL_COLUMN}' was not found.\n"
            f"Available columns: {list(source_df.columns)}"
        )

if SPEECH_COLUMN not in source_df.columns:
    raise KeyError(
        f"Expected Notebook 1's '{SPEECH_COLUMN}' column, but it was not found."
    )

source_df[ID_COLUMN] = source_df[ID_COLUMN].astype(str)

duplicate_mask = source_df[ID_COLUMN].duplicated(keep=False)
if duplicate_mask.any():
    duplicate_examples = (
        source_df.loc[duplicate_mask, ID_COLUMN]
        .head(10)
        .tolist()
    )
    raise ValueError(
        "Duplicate IDs were found in audio_speech_labels.csv. "
        f"Examples: {duplicate_examples}"
    )

print(f"Rows available for YAMNet: {len(source_df):,}")

print("\nExisting speech/non-speech labels from Notebook 1:")
display(
    source_df[SPEECH_COLUMN]
    .value_counts(dropna=False)
    .rename("count")
)

display(source_df.head())


Rows available for YAMNet: 6,374

Existing speech/non-speech labels from Notebook 1:


is_speech
True     4463
False    1911
Name: count, dtype: int64

,id,createdAt,updatedAt,uploadedUrl,streamableUrl,transcodeJobId,transcodeStatus,transcodeMessage,duration,is_speech
0,1dff287a-bf34-4e80-9505-52766d2bfda8,2021-02-09 17:49:14.799697,2021-02-09 17:49:14.799697,https://says-api-uploaded-audio-dev.s3.amazona...,https://says-api-streamable-audio-dev.s3.amazo...,1612892954758-rkkrui,Submitted,NaN,NaN,True
1,d30d00f8-633d-4c85-9f10-e41fc0ea4780,2021-02-09 16:05:37.929757,2021-02-09 16:05:37.929757,https://says-api-uploaded-audio-dev.s3.amazona...,https://says-api-streamable-audio-dev.s3.amazo...,1612886737875-8x74sh,Submitted,NaN,NaN,True
2,4368c30e-a587-483c-86df-749ee160e930,2021-02-08 18:12:12.133277,2021-02-08 18:12:12.133277,https://says-api-uploaded-audio-dev.s3.amazona...,https://says-api-streamable-audio-dev.s3.amazo...,1612807932078-sin4uz,Submitted,NaN,NaN,True
3,da974a87-3062-4084-9f08-387eaa2dd9e0,2021-03-15 04:50:46.875269,2021-03-15 04:55:46.985315,https://says-api-uploaded-audio.s3.amazonaws.c...,https://says-api-streamable-audio.s3.amazonaws...,1615783846760-9zsc2u,Complete,Transcode job 1615783846760-9zsc2u was completed.,6.0,False
4,83ab625b-736c-4ec9-acc9-6f037cd42d84,2021-03-15 04:51:11.921958,2021-03-15 04:56:12.024077,https://says-api-uploaded-audio.s3.amazonaws.c...,https://says-api-streamable-audio.s3.amazonaws...,1615783871838-ix7lro,Complete,Transcode job 1615783871838-ix7lro was completed.,6.0,False


## 5. Locate the cached audio exactly as Notebook 1 does

Notebook 1 derives the cached extension from the streamable URL. This
function reproduces the same logic and also includes a safe fallback search
for an already-downloaded file with the same ID.

In [5]:
SUPPORTED_AUDIO_SUFFIXES = {
    ".mp3",
    ".m4a",
    ".wav",
    ".ogg",
    ".aac",
    ".flac",
}


def cache_path_for(row_id: str, url: str) -> Path:
    """
    Same cache-path rule used by Notebook 1.
    """
    suffix = Path(urlparse(url).path).suffix.lower() or ".mp3"

    if suffix not in SUPPORTED_AUDIO_SUFFIXES:
        suffix = ".mp3"

    return AUDIO_DIR / f"{row_id}{suffix}"


def find_cached_audio(row: pd.Series) -> Path:
    """
    Find the audio downloaded by Notebook 1.

    First uses Notebook 1's exact URL-based path. If that path is not found,
    searches the cache for a supported audio file with the same ID.
    """
    row_id = str(row[ID_COLUMN]).strip()
    url_value = row.get(AUDIO_URL_COLUMN)

    if pd.notna(url_value) and str(url_value).strip():
        expected_path = cache_path_for(
            row_id,
            str(url_value).strip(),
        )

        if expected_path.exists() and expected_path.stat().st_size > 0:
            return expected_path

    fallback_candidates = []

    for suffix in SUPPORTED_AUDIO_SUFFIXES:
        candidate = AUDIO_DIR / f"{row_id}{suffix}"

        if candidate.exists() and candidate.stat().st_size > 0:
            fallback_candidates.append(candidate)

    if len(fallback_candidates) == 1:
        return fallback_candidates[0]

    if len(fallback_candidates) > 1:
        # Prefer the largest complete cached file if multiple formats exist.
        return max(
            fallback_candidates,
            key=lambda path: path.stat().st_size,
        )

    raise FileNotFoundError(
        f"No downloaded audio file found for {ID_COLUMN}={row_id} "
        f"inside {AUDIO_DIR}"
    )


## 6. Decode cached audio for YAMNet

The cached files may be MP3, M4A, WAV, OGG, AAC, or FLAC. FFmpeg converts
each local file directly into the 16 kHz mono waveform required by YAMNet.

No audio is downloaded again.

In [6]:
def decode_audio_for_yamnet(
    audio_path: Path,
    sample_rate: int = TARGET_SAMPLE_RATE,
) -> np.ndarray:
    """
    Decode a local audio file into mono float32 samples at 16 kHz.
    """
    command = [
        "ffmpeg",
        "-nostdin",
        "-hide_banner",
        "-loglevel", "error",
        "-i", str(audio_path),
        "-vn",
        "-ac", "1",
        "-ar", str(sample_rate),
        "-f", "s16le",
        "-",
    ]

    try:
        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            timeout=FFMPEG_TIMEOUT_SECONDS,
            check=False,
        )
    except subprocess.TimeoutExpired as error:
        raise TimeoutError(
            f"ffmpeg timed out while decoding {audio_path.name}"
        ) from error

    if result.returncode != 0:
        error_message = result.stderr.decode(
            "utf-8",
            errors="ignore",
        ).strip()

        raise RuntimeError(
            error_message
            or f"ffmpeg failed to decode {audio_path.name}"
        )

    waveform = (
        np.frombuffer(result.stdout, dtype=np.int16)
        .astype(np.float32)
        / 32768.0
    )

    if waveform.size == 0:
        raise ValueError(
            f"Decoded waveform is empty: {audio_path.name}"
        )

    if waveform.size < sample_rate // 10:
        raise ValueError(
            f"Decoded audio is shorter than 0.1 seconds: {audio_path.name}"
        )

    return np.ascontiguousarray(waveform, dtype=np.float32)


## 7. Load YAMNet and define top-5 inference

Ranks are based on the **mean YAMNet class score across all YAMNet frames**
in the complete clip.

In [7]:
def load_yamnet_class_names(model: Any) -> list[str]:
    """
    Read the 521 class names packaged with the YAMNet model.
    """
    class_map_path = model.class_map_path().numpy()

    if isinstance(class_map_path, bytes):
        class_map_path = class_map_path.decode("utf-8")

    class_names = []

    with tf.io.gfile.GFile(class_map_path) as class_map_file:
        reader = csv.DictReader(class_map_file)

        for row in reader:
            class_names.append(row["display_name"])

    if len(class_names) != 521:
        raise ValueError(
            f"Expected 521 YAMNet classes but loaded {len(class_names)}."
        )

    return class_names


def infer_yamnet_top5(
    model: Any,
    class_names: list[str],
    waveform: np.ndarray,
) -> dict[str, Any]:
    """
    Run YAMNet and return five ranked classes plus their scores.
    """
    scores, _embeddings, _spectrogram = model(
        tf.convert_to_tensor(
            waveform,
            dtype=tf.float32,
        )
    )

    scores_np = scores.numpy()

    if scores_np.ndim != 2:
        raise ValueError(
            f"Unexpected YAMNet score shape: {scores_np.shape}"
        )

    if scores_np.shape[1] != len(class_names):
        raise ValueError(
            "YAMNet score/class-name mismatch: "
            f"{scores_np.shape[1]} scores vs {len(class_names)} names."
        )

    mean_scores = scores_np.mean(axis=0)
    max_scores = scores_np.max(axis=0)

    ranked_indices = np.argsort(mean_scores)[::-1][:TOP_K]

    result: dict[str, Any] = {
        "yamnet_frame_count": int(scores_np.shape[0]),
        "yamnet_audio_seconds_used": float(
            waveform.size / TARGET_SAMPLE_RATE
        ),
    }

    json_result = []

    for rank, class_index in enumerate(
        ranked_indices,
        start=1,
    ):
        class_index = int(class_index)
        class_name = class_names[class_index]
        mean_score = float(mean_scores[class_index])
        max_score = float(max_scores[class_index])

        result[f"yamnet_rank_{rank}_class"] = class_name
        result[f"yamnet_rank_{rank}_mean_score"] = mean_score
        result[f"yamnet_rank_{rank}_max_score"] = max_score

        json_result.append(
            {
                "rank": rank,
                "class_index": class_index,
                "class_name": class_name,
                "mean_score": round(mean_score, 8),
                "max_score": round(max_score, 8),
            }
        )

    result["yamnet_top5_json"] = json.dumps(
        json_result,
        ensure_ascii=False,
    )

    return result


In [8]:
print("Loading YAMNet from TensorFlow Hub...")
print("The first run may download the model once.")

yamnet_model = hub.load(YAMNET_HANDLE)
yamnet_class_names = load_yamnet_class_names(yamnet_model)

print(
    f"YAMNet ready: {len(yamnet_class_names)} classes loaded."
)


Loading YAMNet from TensorFlow Hub...
The first run may download the model once.
YAMNet ready: 521 classes loaded.


## 8. Checkpoint/resume helpers

The new CSV is saved atomically during the run. If the notebook is
interrupted, rerunning it skips clips already marked `completed`.

In [9]:
def load_existing_yamnet_output() -> pd.DataFrame:
    if not OUTPUT_CSV.exists():
        return pd.DataFrame()

    try:
        existing = pd.read_csv(OUTPUT_CSV)

        if ID_COLUMN in existing.columns:
            existing[ID_COLUMN] = existing[ID_COLUMN].astype(str)

        return existing

    except Exception as error:
        raise RuntimeError(
            f"Could not read existing YAMNet CSV {OUTPUT_CSV}: {error}"
        ) from error


def existing_records_by_id(
    existing_df: pd.DataFrame,
) -> dict[str, dict[str, Any]]:
    """
    Convert existing YAMNet output into resumable result records.
    """
    if existing_df.empty or ID_COLUMN not in existing_df.columns:
        return {}

    yamnet_columns = [
        column
        for column in existing_df.columns
        if column.startswith("yamnet_")
    ]

    records: dict[str, dict[str, Any]] = {}

    for _, row in existing_df.iterrows():
        row_id = str(row[ID_COLUMN])
        record = {ID_COLUMN: row_id}

        for column in yamnet_columns:
            value = row.get(column)

            if pd.notna(value):
                record[column] = value

        records[row_id] = record

    return records


def save_yamnet_csv(
    original_df: pd.DataFrame,
    yamnet_records: dict[str, dict[str, Any]],
) -> pd.DataFrame:
    """
    Write a NEW CSV containing all original Notebook 1 columns plus
    the YAMNet result columns.

    audio_speech_labels.csv is never modified.
    """
    original = original_df.copy()
    original[ID_COLUMN] = original[ID_COLUMN].astype(str)

    results_df = pd.DataFrame(
        yamnet_records.values()
    )

    if results_df.empty:
        merged = original.copy()
    else:
        results_df[ID_COLUMN] = results_df[ID_COLUMN].astype(str)

        yamnet_columns = [
            column
            for column in results_df.columns
            if column != ID_COLUMN
        ]

        # Avoid duplicate YAMNet columns if this function is rerun
        # against a dataframe that already contains them.
        original = original.drop(
            columns=yamnet_columns,
            errors="ignore",
        )

        merged = original.merge(
            results_df,
            on=ID_COLUMN,
            how="left",
            validate="one_to_one",
        )

    temporary_csv = OUTPUT_CSV.with_suffix(
        OUTPUT_CSV.suffix + ".tmp"
    )

    merged.to_csv(
        temporary_csv,
        index=False,
    )

    os.replace(
        temporary_csv,
        OUTPUT_CSV,
    )

    return merged


## 9. Run YAMNet over the complete classified dataset

This processes **both speech and non-speech rows**. The existing `is_speech`
column is simply carried into the new output CSV.

In [10]:
existing_output = load_existing_yamnet_output()
yamnet_records = existing_records_by_id(existing_output)

pending_rows = []

for _, row in source_df.iterrows():
    row_id = str(row[ID_COLUMN])

    previous_status = yamnet_records.get(
        row_id,
        {},
    ).get("yamnet_status")

    if previous_status == "completed":
        continue

    if (
        previous_status == "failed"
        and not RETRY_FAILED
    ):
        continue

    pending_rows.append(row)

if LIMIT is not None:
    pending_rows = pending_rows[:max(0, int(LIMIT))]

already_completed = sum(
    record.get("yamnet_status") == "completed"
    for record in yamnet_records.values()
)

print(f"Rows in source CSV:     {len(source_df):,}")
print(f"Already completed:      {already_completed:,}")
print(f"Selected for this run:  {len(pending_rows):,}")
print(f"Output CSV:             {OUTPUT_CSV}")

processed_this_run = 0
completed_this_run = 0
failed_this_run = 0

start_time = time.monotonic()

try:
    for row in tqdm(
        pending_rows,
        total=len(pending_rows),
        desc="YAMNet top-5",
        unit="file",
    ):
        row_id = str(row[ID_COLUMN])

        record: dict[str, Any] = {
            ID_COLUMN: row_id,
            "yamnet_processed_at_utc": pd.Timestamp.now(
                tz="UTC"
            ).isoformat(),
        }

        try:
            audio_path = find_cached_audio(row)

            waveform = decode_audio_for_yamnet(
                audio_path
            )

            prediction = infer_yamnet_top5(
                model=yamnet_model,
                class_names=yamnet_class_names,
                waveform=waveform,
            )

            record.update(prediction)

            record["yamnet_audio_path"] = str(audio_path)
            record["yamnet_status"] = "completed"
            record["yamnet_error"] = ""

            completed_this_run += 1

        except Exception as error:
            record["yamnet_status"] = "failed"
            record["yamnet_error"] = (
                f"{type(error).__name__}: {error}"
            )[:4000]

            failed_this_run += 1

            print(
                f"\n[YAMNET FAILED] "
                f"{ID_COLUMN}={row_id}: "
                f"{record['yamnet_error']}"
            )

        yamnet_records[row_id] = record
        processed_this_run += 1

        if (
            processed_this_run % SAVE_EVERY == 0
        ):
            save_yamnet_csv(
                source_df,
                yamnet_records,
            )

except KeyboardInterrupt:
    print(
        "\nInterrupted. Saving all completed YAMNet progress..."
    )

    save_yamnet_csv(
        source_df,
        yamnet_records,
    )

    raise

final_df = save_yamnet_csv(
    source_df,
    yamnet_records,
)

elapsed_minutes = (
    time.monotonic() - start_time
) / 60.0

print("\nYAMNet stage finished.")
print(f"Processed this run: {processed_this_run:,}")
print(f"Completed this run: {completed_this_run:,}")
print(f"Failed this run:    {failed_this_run:,}")
print(f"Elapsed:            {elapsed_minutes:.1f} minutes")
print(f"NEW CSV saved to:   {OUTPUT_CSV}")


Rows in source CSV:     6,374
Already completed:      0
Selected for this run:  6,374
Output CSV:             /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/yamnet_top5_classifications.csv


YAMNet top-5:   0%|          | 0/6374 [00:00<?, ?file/s]


[YAMNET FAILED] id=1f7b6576-3ac8-4c98-b13d-73f73f8f0d4f: ValueError: Decoded audio is shorter than 0.1 seconds: 1f7b6576-3ac8-4c98-b13d-73f73f8f0d4f.mp3

[YAMNET FAILED] id=e38993c9-3ba0-4ec3-938b-a61eaa1ca745: ValueError: Decoded audio is shorter than 0.1 seconds: e38993c9-3ba0-4ec3-938b-a61eaa1ca745.mp3

[YAMNET FAILED] id=900151a3-f4ff-47f1-91cf-a25e0dab2b04: ValueError: Decoded audio is shorter than 0.1 seconds: 900151a3-f4ff-47f1-91cf-a25e0dab2b04.mp3

[YAMNET FAILED] id=91ff2909-c571-4610-aec7-3b56b0134396: ValueError: Decoded audio is shorter than 0.1 seconds: 91ff2909-c571-4610-aec7-3b56b0134396.mp3

[YAMNET FAILED] id=fcac00be-7f11-4da8-83e6-ce944e11fa35: ValueError: Decoded audio is shorter than 0.1 seconds: fcac00be-7f11-4da8-83e6-ce944e11fa35.mp3

YAMNet stage finished.
Processed this run: 6,374
Completed this run: 6,369
Failed this run:    5
Elapsed:            57.0 minutes
NEW CSV saved to:   /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/yamnet_top5_

## 10. Verify the top-5 output

This checks that completed clips have all five YAMNet class columns and shows
a compact preview.

In [11]:
result_df = pd.read_csv(OUTPUT_CSV)

top5_class_columns = [
    f"yamnet_rank_{rank}_class"
    for rank in range(1, TOP_K + 1)
]

preview_columns = [
    ID_COLUMN,
    SPEECH_COLUMN,
    "yamnet_status",
]

for rank in range(1, TOP_K + 1):
    preview_columns.extend(
        [
            f"yamnet_rank_{rank}_class",
            f"yamnet_rank_{rank}_mean_score",
        ]
    )

preview_columns = [
    column
    for column in preview_columns
    if column in result_df.columns
]

display(
    result_df[
        preview_columns
    ].head(20)
)

print("\nYAMNet processing status:")
display(
    result_df["yamnet_status"]
    .value_counts(dropna=False)
    .rename("count")
)

print("\nSpeech/non-speech labels retained from Notebook 1:")
display(
    result_df[SPEECH_COLUMN]
    .value_counts(dropna=False)
    .rename("count")
)

completed_df = result_df[
    result_df["yamnet_status"].eq("completed")
].copy()

if all(
    column in completed_df.columns
    for column in top5_class_columns
):
    incomplete_count = int(
        completed_df[top5_class_columns]
        .isna()
        .any(axis=1)
        .sum()
    )

    print(
        "\nCompleted rows missing one or more top-5 classes:",
        incomplete_count,
    )
else:
    missing_columns = [
        column
        for column in top5_class_columns
        if column not in completed_df.columns
    ]

    print(
        "\nMissing expected YAMNet columns:",
        missing_columns,
    )

print(f"\nFinal YAMNet CSV: {OUTPUT_CSV.resolve()}")


,id,is_speech,yamnet_status,yamnet_rank_1_class,yamnet_rank_1_mean_score,yamnet_rank_2_class,yamnet_rank_2_mean_score,yamnet_rank_3_class,yamnet_rank_3_mean_score,yamnet_rank_4_class,yamnet_rank_4_mean_score,yamnet_rank_5_class,yamnet_rank_5_mean_score
0,1dff287a-bf34-4e80-9505-52766d2bfda8,True,completed,Speech,0.751908,Vehicle,3.719240e-02,Motor vehicle (road),1.987645e-02,Car,1.956571e-02,Music,1.897684e-02
1,d30d00f8-633d-4c85-9f10-e41fc0ea4780,True,completed,Speech,0.392640,Vehicle,5.005489e-02,Motor vehicle (road),3.156682e-02,Child singing,2.810976e-02,Car,2.496984e-02
2,4368c30e-a587-483c-86df-749ee160e930,True,completed,Speech,0.392640,Vehicle,5.005489e-02,Motor vehicle (road),3.156682e-02,Child singing,2.810976e-02,Car,2.496984e-02
3,da974a87-3062-4084-9f08-387eaa2dd9e0,False,completed,Silence,1.000000,Speech,3.329521e-09,Music,3.726664e-17,"Inside, small room",7.226350e-23,"Narration, monologue",2.736002e-25
4,83ab625b-736c-4ec9-acc9-6f037cd42d84,False,completed,Silence,1.000000,Speech,4.451892e-08,Music,4.410476e-15,"Inside, small room",2.154140e-21,"Narration, monologue",6.826833e-22
5,c0fe394b-6f0e-4cb1-934b-52248c1fa505,False,completed,Silence,1.000000,Speech,4.451892e-08,Music,4.410476e-15,"Inside, small room",2.154140e-21,"Narration, monologue",6.826833e-22
6,a7c269ed-f38b-4f15-a7b2-b692cdb53a29,False,completed,Silence,0.848612,Speech,4.809593e-02,"Narration, monologue",6.488358e-03,"Inside, small room",3.894899e-03,Chant,3.162292e-03
7,ca6f5410-ca34-42bc-a16d-b76d78e82435,False,completed,Silence,0.669941,Speech,1.383491e-01,"Inside, small room",1.276422e-02,"Burping, eructation",4.657222e-03,Clicking,2.477663e-03
8,ad49e621-f522-4ff2-a665-2c2fbad16fc2,False,completed,Silence,0.981029,Speech,4.349487e-03,"Inside, small room",5.145336e-04,"Narration, monologue",4.512371e-04,"Inside, large room or hall",2.985283e-04
9,6bc7b6cd-ff35-434b-b9f9-8fb1105a0cf3,False,completed,Silence,0.571118,Speech,4.232588e-02,Animal,2.873757e-02,Wild animals,2.696374e-02,Owl,2.631536e-02



YAMNet processing status:


yamnet_status
completed    6369
failed          5
Name: count, dtype: int64


Speech/non-speech labels retained from Notebook 1:


is_speech
True     4463
False    1911
Name: count, dtype: int64


Completed rows missing one or more top-5 classes: 0

Final YAMNet CSV: /Users/tavishikaushik/Downloads/SaysUserContentRecommendation/data/yamnet_top5_classifications.csv


## 11. Optional — inspect speech and non-speech separately

This does not change any labels. It only filters the finished YAMNet CSV
using Notebook 1's existing `is_speech` value.

In [12]:
speech_yamnet = result_df[
    result_df[SPEECH_COLUMN].eq(True)
].copy()

non_speech_yamnet = result_df[
    result_df[SPEECH_COLUMN].eq(False)
].copy()

print(f"Speech rows:     {len(speech_yamnet):,}")
print(f"Non-speech rows: {len(non_speech_yamnet):,}")

print("\nSpeech sample:")
display(
    speech_yamnet[
        preview_columns
    ].head(5)
)

print("\nNon-speech sample:")
display(
    non_speech_yamnet[
        preview_columns
    ].head(5)
)


Speech rows:     4,463
Non-speech rows: 1,911

Speech sample:


,id,is_speech,yamnet_status,yamnet_rank_1_class,yamnet_rank_1_mean_score,yamnet_rank_2_class,yamnet_rank_2_mean_score,yamnet_rank_3_class,yamnet_rank_3_mean_score,yamnet_rank_4_class,yamnet_rank_4_mean_score,yamnet_rank_5_class,yamnet_rank_5_mean_score
0,1dff287a-bf34-4e80-9505-52766d2bfda8,True,completed,Speech,0.751908,Vehicle,0.037192,Motor vehicle (road),0.019876,Car,0.019566,Music,0.018977
1,d30d00f8-633d-4c85-9f10-e41fc0ea4780,True,completed,Speech,0.392640,Vehicle,0.050055,Motor vehicle (road),0.031567,Child singing,0.028110,Car,0.024970
2,4368c30e-a587-483c-86df-749ee160e930,True,completed,Speech,0.392640,Vehicle,0.050055,Motor vehicle (road),0.031567,Child singing,0.028110,Car,0.024970
10,84cb5b23-465f-44fc-8366-5913a9350efb,True,completed,Speech,0.392640,Vehicle,0.050055,Motor vehicle (road),0.031567,Child singing,0.028110,Car,0.024970
16,f11559d7-d8ac-4e78-9868-c3352e02ce66,True,completed,Speech,0.909672,Silence,0.057200,"Narration, monologue",0.014120,"Inside, small room",0.014051,"Burping, eructation",0.005663



Non-speech sample:


,id,is_speech,yamnet_status,yamnet_rank_1_class,yamnet_rank_1_mean_score,yamnet_rank_2_class,yamnet_rank_2_mean_score,yamnet_rank_3_class,yamnet_rank_3_mean_score,yamnet_rank_4_class,yamnet_rank_4_mean_score,yamnet_rank_5_class,yamnet_rank_5_mean_score
3,da974a87-3062-4084-9f08-387eaa2dd9e0,False,completed,Silence,1.000000,Speech,3.329521e-09,Music,3.726664e-17,"Inside, small room",7.226350e-23,"Narration, monologue",2.736002e-25
4,83ab625b-736c-4ec9-acc9-6f037cd42d84,False,completed,Silence,1.000000,Speech,4.451892e-08,Music,4.410476e-15,"Inside, small room",2.154140e-21,"Narration, monologue",6.826833e-22
5,c0fe394b-6f0e-4cb1-934b-52248c1fa505,False,completed,Silence,1.000000,Speech,4.451892e-08,Music,4.410476e-15,"Inside, small room",2.154140e-21,"Narration, monologue",6.826833e-22
6,a7c269ed-f38b-4f15-a7b2-b692cdb53a29,False,completed,Silence,0.848612,Speech,4.809593e-02,"Narration, monologue",6.488358e-03,"Inside, small room",3.894899e-03,Chant,3.162292e-03
7,ca6f5410-ca34-42bc-a16d-b76d78e82435,False,completed,Silence,0.669941,Speech,1.383491e-01,"Inside, small room",1.276422e-02,"Burping, eructation",4.657222e-03,Clicking,2.477663e-03
